# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vaibhavrajput326/flyrank.ai/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*States clearly that one row is daily aggregated performance for a specific content_hash_id and client_hash_id on a given report_date

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
from google.colab import userdata

# Retrieve Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')

# Initialize DuckDB connection & set Hugging Face Secret
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, Token '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
One Row: Daily aggregated search and traffic performance for a single content item (content_hash_id) belonging to a client (client_hash_id) on a given date (report_date).

Table(s): fact_content_daily_performance (partitioned by month).

Time Window: Mid-panel month 2026-03 for training, keeping 2026-06 sealed as the test set.

Label / Target: is_decayed (binary flag indicating if search clicks drop significantly in late month).

Excluded: Future impressions/clicks post-decision date and unverified crawler/bot sessions.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q1 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (client_hash_id || '_' || content_hash_id || '_' || CAST(report_date AS VARCHAR))) AS unique_grain_count
FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet', HIVE_PARTITIONING=TRUE)
WHERE month = '2026-03'
"""
df1 = con.sql(q1).df()
print("Query 1 Result (Grain Verification):")
display(df1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 Result (Grain Verification):


,total_rows,unique_grain_count
0,9841378,9841378


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
Training on mid-panel month 2026-03 (and keeping 2026-06 sealed as the test month)

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q2 = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(DISTINCT content_hash_id) AS unique_pages,
    COUNT(DISTINCT client_hash_id) AS unique_clients
FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet', HIVE_PARTITIONING=TRUE)
WHERE month = '2026-03'
"""
df2 = con.sql(q2).df()
print("Query 2 Result (Row Count & Date Span):")
display(df2)

Query 2 Result (Row Count & Date Span):


,total_rows,start_date,end_date,unique_pages,unique_clients
0,9841378,2026-03-01,2026-03-31,331437,55


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
Named Data Limitation: “Unbalanced Panel & Search Anonymization Tail Risk: The slice relies on fact_content_daily_performance for month=2026-03. Because client history onboarding dates vary and Google Search Console suppresses long-tail search queries (< 10 impressions), low-traffic pages exhibit zero-inflated signals that do not reflect true user demand.”

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q3 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(CASE WHEN client_has_gsc IS TRUE THEN 1 END) AS available_survived_rows,
    ROUND(COUNT(CASE WHEN client_has_gsc IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) AS survival_pct
FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet', HIVE_PARTITIONING=TRUE)
WHERE month = '2026-03'
"""
df3 = con.sql(q3).df()
print("Query 3 Result (Availability Filter with IS TRUE):")
display(df3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 3 Result (Availability Filter with IS TRUE):


,total_rows,available_survived_rows,survival_pct
0,9841378,9841378,100.0


In [8]:
feature_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    -- 5 HONEST FEATURES (Historical)
    SUM(gsc_impressions) AS gsc_impressions_30d,
    SUM(gsc_clicks) AS gsc_clicks_30d,
    SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS historical_ctr,
    AVG(gsc_avg_position) AS avg_position_30d,
    MAX(report_date) - MIN(report_date) AS observation_window_days,

    -- LEAKED FEATURE (The Trap: Future performance metric)
    SUM(CASE WHEN report_date > '2026-03-20' THEN gsc_clicks ELSE 0 END) AS LEAKED_future_clicks,

    -- HONEST GROUND TRUTH LABEL
    CASE WHEN SUM(CASE WHEN report_date > '2026-03-20' THEN gsc_clicks ELSE 0 END) < 10 THEN 1 ELSE 0 END AS is_decayed
FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet', HIVE_PARTITIONING=TRUE)
WHERE month = '2026-03' AND client_has_gsc IS TRUE
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) > 50
"""

5-Feature Frame & Decision Moment Justifications:

gsc_impressions_30d: Knowable at the decision moment because it aggregates historical impressions prior to decision date.

gsc_clicks_30d: Knowable at the decision moment because it relies strictly on past search click logs.

historical_ctr: Knowable at the decision moment because it is derived purely from historical clicks and impressions.

avg_position_30d: Knowable at the decision moment because it computes average SERP ranking over the observation period.

observation_window_days: Knowable at the decision moment because the active date span is fixed prior to inference.

In [9]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1. Fetch feature dataframe from DuckDB
df_features = con.sql(feature_query).df().fillna(0)

# Ensure numeric types
numeric_cols = [
    'gsc_impressions_30d', 'gsc_clicks_30d', 'historical_ctr',
    'avg_position_30d', 'observation_window_days',
    'LEAKED_future_clicks', 'is_decayed'
]
for col in numeric_cols:
    if col in df_features.columns:
        df_features[col] = pd.to_numeric(df_features[col])

# Convert timedelta to integer if observation_window_days is returned as a duration
if pd.api.types.is_timedelta64_dtype(df_features['observation_window_days']):
    df_features['observation_window_days'] = df_features['observation_window_days'].dt.days

# 2. Define Feature Sets
honest_features = [
    'gsc_impressions_30d', 'gsc_clicks_30d', 'historical_ctr',
    'avg_position_30d', 'observation_window_days'
]
leaked_features = honest_features + ['LEAKED_future_clicks']

X_honest = df_features[honest_features]
X_leaked = df_features[leaked_features]
y = df_features['is_decayed']

# 3. Model WITH Leaked Feature (The Trap)
clf_leaked = RandomForestClassifier(n_estimators=50, random_state=42)
clf_leaked.fit(X_leaked, y)
score_leaked = roc_auc_score(y, clf_leaked.predict_proba(X_leaked)[:, 1])

# 4. Model WITHOUT Leaked Feature (Honest Model)
clf_honest = RandomForestClassifier(n_estimators=50, random_state=42)
clf_honest.fit(X_honest, y)
score_honest = roc_auc_score(y, clf_honest.predict_proba(X_honest)[:, 1])

print(f"🚨 Score WITH Leaked Feature (Fake Perfection): ROC-AUC = {score_leaked:.4f}")
print(f"✅ Score WITHOUT Leaked Feature (Honest Baseline): ROC-AUC = {score_honest:.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

🚨 Score WITH Leaked Feature (Fake Perfection): ROC-AUC = 1.0000
✅ Score WITHOUT Leaked Feature (Honest Baseline): ROC-AUC = 1.0000


🚨 Score WITH Leaked Feature (Fake Perfection): ROC-AUC = 1.0000
✅ Score WITHOUT Leaked Feature (Honest Baseline): ROC-AUC = 1.0000
Explicitly excludes post-decision future clicks/impressions and unverified bot sessions

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.